# LoRA/QLoRA Fine-tuning — End-to-End Walkthrough

This notebook walks through the complete pipeline:
1. Environment setup & dependencies
2. Load & inspect a domain dataset
3. Load base model with 4-bit QLoRA quantization
4. Attach LoRA adapters & inspect trainable parameters
5. Train with SFTTrainer
6. Evaluate: perplexity, ROUGE, BERTScore
7. Merge adapter to standalone model
8. Interactive inference demo

## 1. Install Dependencies

In [ ]:
import os
os.chdir('..')
os.getcwd()

In [ ]:
# Run once
# !pip install -q transformers datasets peft trl bitsandbytes accelerate
# !pip install -q rouge-score bert-score wandb rich
import sys
print(f'Python {sys.version}')

import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Dataset Exploration

In [ ]:
from datasets import load_dataset
import pandas as pd

# Using the Alpaca dataset as a universal demo
dataset = load_dataset('tatsu-lab/alpaca', split='train')
print(f'Dataset size: {len(dataset):,} samples')
print(f'Columns: {dataset.column_names}')
dataset.to_pandas().head(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analyse instruction/output length distributions
df = dataset.to_pandas()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['instruction'].str.len().hist(bins=60, ax=axes[0], color='#4f86c6', edgecolor='white')
axes[0].set_title('Instruction Length Distribution', fontsize=13)
axes[0].set_xlabel('Characters')

df['output'].str.len().hist(bins=60, ax=axes[1], color='#e07b54', edgecolor='white')
axes[1].set_title('Output Length Distribution', fontsize=13)
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.show()
print(f"Instruction — mean: {df['instruction'].str.len().mean():.0f} | min: {df['instruction'].str.len().min()} | max: {df['instruction'].str.len().max()}")
print(f"Output      — mean: {df['output'].str.len().mean():.0f} | min:{df['output'].str.len().min()} | max: {df['output'].str.len().max()}")

## 3. Prepare Data — Alpaca Instruction Format

In [ ]:
import sys; sys.path.append('./src')
from src.data_utils import text_to_alpaca, train_val_test_split, mask_prompt_tokens
from datasets import Dataset

INSTRUCTION_TEMPLATE = '### Instruction:\n'
RESPONSE_TEMPLATE = '### Response:\n'

def format_sample(row):
    return {'text': text_to_alpaca(row)}

formatted = dataset.map(format_sample, num_proc=4)
splits = train_val_test_split(formatted, val_ratio=0.05, test_ratio=0.05)
print(splits)
print('\n--- Example formatted sample ---')
print(splits['train'][0]['text'][:500])

## 4. Load Model with QLoRA (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = 'mistralai/Mistral-7B-v0.1'  # swap for any CausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='right')
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.bfloat16,
)
base_model = prepare_model_for_kbit_training(base_model)
print('>>>>Base model loaded with 4-bit quantization')

## 5. Attach LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=64,                  # rank — higher = more capacity, more params
    lora_alpha=16,         # scaling factor
    target_modules=[       # which weight matrices to adapt
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Visualise adapter parameter distribution
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params:     {total/1e9:.2f}B')
print(f'Trainable params: {trainable/1e6:.2f}M  ({100*trainable/total:.3f}%)')

## 6. Train with SFTTrainer

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl.trainer.sft_trainer import SFTTrainer
from src.data_utils import mask_prompt_tokens
import torch
#from trl.trainer import DataCollatorForCompletionOnlyLM

OUTPUT_DIR = './outputs/alpaca-lora-demo'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,               # increase for production
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=3,
    bf16=True,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    optim='paged_adamw_32bit',
    max_grad_norm=0.3,
  #  group_by_length=True,
    report_to='none',  # set to 'wandb' for experiment tracking
)

collator = mask_prompt_tokens(tokenizer=tokenizer,
    prompt_template=INSTRUCTION_TEMPLATE,
    response_template=RESPONSE_TEMPLATE,
)

trainer = SFTTrainer(
    model=model,
  #  tokenizer=tokenizer,
    train_dataset=splits['train'],
    eval_dataset=splits['validation'],
  #  dataset_text_field='text',
    data_collator=collator,
 #   max_seq_length=2048,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## 7. Evaluate: Loss Curve + Perplexity

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_steps = [l['step'] for l in logs if 'loss' in l and 'eval_loss' not in l]
train_loss  = [l['loss'] for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_steps  = [l['step'] for l in logs if 'eval_loss' in l]
eval_loss   = [l['eval_loss'] for l in logs if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train_steps, train_loss, label='Train Loss', color='#4f86c6', linewidth=2)
ax.plot(eval_steps, eval_loss, label='Val Loss', color='#e07b54', linewidth=2, linestyle='--')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('LoRA Fine-tuning Loss Curve', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./docs/loss_curve.png', dpi=150)
plt.show()

In [ ]:
import sys; sys.path.append('../src')
from evaluate import compute_perplexity, compute_rouge

test_texts = [splits['test'][i]['text'] for i in range(min(50, len(splits['test'])))]
ppl_results = compute_perplexity(model, tokenizer, test_texts[:20])
print('Perplexity on test set:')
for k, v in ppl_results.items():
    print(f'  {k}: {v}')

## 8. Save & Merge Adapter

In [ ]:
# Save adapter only (small — ~100MB for r=64)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

# Optional: merge into full model (for deployment)
# from inference import merge_and_save
# merge_and_save(MODEL_NAME, OUTPUT_DIR, '../outputs/alpaca-merged')

## 9. Inference Demo

In [ ]:
from peft import PeftModel
import torch

model.eval()

def generate(prompt, max_new_tokens=256, temperature=0.7):
    full_prompt = f'### Instruction:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(full_prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

# --- Try it out ---
test_prompts = [
    'Explain the difference between supervised and unsupervised learning.',
    'Write a Python function that checks if a string is a palindrome.',
    'What are the key principles of SOLID software design?',
]

for prompt in test_prompts:
    print(f'\n>>> Prompt: {prompt}')
    print(f'>>> Response: {generate(prompt)}')
    print('-' * 70)